# 📓 Notebook 1: Document Ingestion

**Purpose**: Take a PDF or EPUB file → sanity check → extract structured Markdown text → identify chapters/sections → save as `book_structure.json`.

**Pipeline**:
```
Input file → Detect type → 🛑 SANITY CHECK → PDF or EPUB extraction → Chapter detection → Quality validation → Save
```

**Output**: `data/intermediate/book_structure.json` — a list of chapter dicts ready for chunking.

> ⚠️ The sanity check will **abort** if the document falls into the "20% we don't handle" category (scanned PDFs, math-heavy, non-English, DRM, table-heavy, image-heavy).

In [ ]:
# ── Imports & Configuration ─────────────────────────────────────────────────
import sys
import json
from pathlib import Path

# Add project root to path so we can import our modules
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import INPUT_DIR, INTERMEDIATE_DIR, BOOK_STRUCTURE_FILE
from src.pdf_parser import extract_pdf_to_markdown, detect_chapters, validate_extraction
from src.epub_parser import extract_epub_to_markdown, parse_epub_chapters, validate_epub_extraction
from src.sanity_check import run_sanity_check

In [ ]:
# ── Set Input File ───────────────────────────────────────────────────────────
# Place your PDF or EPUB in data/input/ and set the filename here:
INPUT_FILE = INPUT_DIR / "your_book.pdf"   # ← CHANGE THIS

assert INPUT_FILE.exists(), f"File not found: {INPUT_FILE}"
file_type = INPUT_FILE.suffix.lower()
print(f"📄 Input: {INPUT_FILE.name}  |  Type: {file_type}")

In [ ]:
# ── 🛑 SANITY CHECK ─────────────────────────────────────────────────────────
# Detects documents we can't handle well and aborts BEFORE wasting compute.
# Checks for: scanned PDFs, math-heavy, image-heavy, non-English,
#              DRM-protected EPUBs, table/data-heavy documents.
#
# If the check FAILS, execution stops here with a clear explanation.
# If it PASSES (possibly with warnings), we continue to extraction.

sanity_result = run_sanity_check(INPUT_FILE, exit_on_fail=True)

In [ ]:
# ── Extract Text ─────────────────────────────────────────────────────────────
# Routes to the appropriate parser based on file extension.
# (Only reached if sanity check passed)

if file_type == ".pdf":
    print("🔍 Extracting PDF with pymupdf4llm...")
    markdown_text = extract_pdf_to_markdown(INPUT_FILE)
    chapters = detect_chapters(markdown_text)
    
elif file_type == ".epub":
    print("🔍 Extracting EPUB with ebooklib...")
    chapters = parse_epub_chapters(INPUT_FILE)
    
else:
    raise ValueError(f"Unsupported file type: {file_type}. Use .pdf or .epub")

print(f"✅ Extracted {len(chapters)} chapter(s)")

In [ ]:
# ── Quality Validation ───────────────────────────────────────────────────────
# Check for garbled text, missing content, or structural issues.

if file_type == ".pdf":
    report = validate_extraction(chapters, INPUT_FILE)
else:
    report = validate_epub_extraction(chapters)

print(f"\n📊 Validation Report:")
print(f"   Status:  {'✅ OK' if report['is_ok'] else '⚠️  Issues detected'}")
if report.get('warnings'):
    for w in report['warnings']:
        print(f"   ⚠️  {w}")

In [ ]:
# ── Preview Extracted Chapters ───────────────────────────────────────────────
# Show the first 300 chars of each chapter so you can visually verify quality.

for i, ch in enumerate(chapters):
    print(f"\n{'='*60}")
    print(f"📖 [{i+1}/{len(chapters)}] {ch.get('chapter', 'Untitled')}")
    print(f"   Section: {ch.get('section', '—')}")
    print(f"   Length:  {len(ch['text']):,} chars")
    print(f"   Preview: {ch['text'][:300]}...")

In [ ]:
# ── Save Output ──────────────────────────────────────────────────────────────
# Save the structured chapters to JSON for the next notebook.

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

with open(BOOK_STRUCTURE_FILE, "w", encoding="utf-8") as f:
    json.dump(chapters, f, ensure_ascii=False, indent=2)

print(f"💾 Saved {len(chapters)} chapters → {BOOK_STRUCTURE_FILE}")
print(f"\n✅ Ingestion complete. Proceed to 02_chunk.ipynb")